# Laboratorio #6 - Etiquetado gramatical y reconocimiento de entidades
Natural Language Processing - UFM

Dataset: Gutenberg Classic Novels Text Dataset (jedidahwavinya/gutenberg-classic-novels-text-dataset).

| # | Título | Autor(a) | Publicación |
|---|---|---|---|
| 1342 | Pride and Prejudice | Jane Austen | 1813 |
| 84 | Frankenstein | Mary Shelley | 1818 |
| 345 | Dracula | Bram Stoker | 1897 |

Las tres son del siglo XIX pero cubren 84 años y sus estilos son distintos: novela de costumbres, novela epistolar filosófica y novela de terror. Eso permite comparar cómo se comporta un tagger entrenado con inglés contemporáneo frente a cada una.

In [ ]:
import os, random, warnings
from collections import Counter, defaultdict

import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import spacy
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_colwidth", 80)

nlp = spacy.load("en_core_web_sm")
print("spaCy", spacy.__version__, "| modelo en_core_web_sm cargado")

In [ ]:
ruta = kagglehub.dataset_download("jedidahwavinya/gutenberg-classic-novels-text-dataset")
df_libros = pd.read_csv(os.path.join(ruta, "gutenberg_novels_dataset.csv"))

ANIOS = {"Pride and Prejudice": 1813, "Frankenstein": 1818, "Dracula": 1897}

INICIO = {
    "Pride and Prejudice": "It is a truth universally acknowledged",
    "Frankenstein": "You will rejoice to hear that no disaster",
    "Dracula": "3 May. Bistritz.",
}


def limpiar_gutenberg(texto, titulo):
    ini = texto.find("*** START")
    if ini != -1:
        texto = texto[texto.find("\n", ini) + 1:]
    fin = texto.find("*** END")
    if fin != -1:
        texto = texto[:fin]

    marca = texto.find(INICIO[titulo])
    if marca != -1:
        texto = texto[marca:]
    return texto.strip()


libros = {}
for _, fila in df_libros.iterrows():
    t = fila["title"]
    libros[t] = {"autor": fila["author"], "anio": ANIOS[t],
                 "texto": limpiar_gutenberg(fila["text"], t)}

for titulo, info in libros.items():
    print(f"{titulo:<22} {info['autor']:<15} {info['anio']}   "
          f"{len(info['texto']):>9,} caracteres")
    print(f"   arranca en: {info['texto'][:70]!r}")

## 1. Preparación de la muestra de trabajo

Mismo pipeline del Laboratorio #5 aplicado a cada novela por separado. De cada una se toman 100 oraciones al azar, con semilla fija.

In [ ]:
random.seed(42)
N_MUESTRA = 100

for titulo, info in libros.items():
    oraciones = sent_tokenize(info["texto"], language="english")
    oraciones = [o for o in oraciones if len(o.split()) >= 4]
    info["oraciones"] = oraciones
    info["tokens_libro"] = sum(len(word_tokenize(o)) for o in oraciones)
    info["muestra"] = random.sample(oraciones, N_MUESTRA)
    info["tokens_muestra"] = sum(len(word_tokenize(o)) for o in info["muestra"])

tabla_muestra = pd.DataFrame([
    {"libro": t, "autor": i["autor"], "año": i["anio"],
     "oraciones (libro)": len(i["oraciones"]), "tokens (libro)": i["tokens_libro"],
     "oraciones (muestra)": len(i["muestra"]), "tokens (muestra)": i["tokens_muestra"],
     "tokens/oración (libro)": round(i["tokens_libro"] / len(i["oraciones"]), 1)}
    for t, i in libros.items()
]).set_index("libro")

tabla_muestra

## 2. Etiquetado gramatical (POS tagging)

Se etiqueta cada token de las tres muestras con en_core_web_sm. Se reportan porcentajes y no conteos, porque las muestras tienen distinto tamaño.

In [ ]:
for titulo, info in libros.items():
    docs = list(nlp.pipe(info["muestra"]))
    info["docs_muestra"] = docs
    info["pos"] = Counter(t.pos_ for d in docs for t in d if not t.is_space)

principales = ["NOUN", "VERB", "ADJ", "ADV", "PRON", "PROPN", "DET", "ADP", "AUX", "PUNCT"]

dist_pos = pd.DataFrame({
    t: {p: info["pos"][p] / sum(info["pos"].values()) * 100 for p in principales}
    for t, info in libros.items()
}).round(2)

dist_pos.loc["(otros)"] = 100 - dist_pos.sum()
dist_pos.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
dist_pos.drop("(otros)").plot(kind="bar", ax=ax,
                              color=["steelblue", "seagreen", "indianred"])
ax.set_ylabel("% de los tokens de la muestra")
ax.set_xlabel("")
ax.set_title("Distribución de categorías gramaticales por novela")
ax.tick_params(axis="x", rotation=0)
ax.grid(axis="y", alpha=.3)
ax.legend(title="")
plt.tight_layout()
plt.show()

In [ ]:
usos = defaultdict(list)
for titulo, info in libros.items():
    for d in info["docs_muestra"]:
        for t in d:
            if t.is_alpha and len(t.text) > 2:
                usos[t.text.lower()].append((t.pos_, t.sent.text.strip(), titulo))

ambiguas = {w: v for w, v in usos.items() if len({p for p, _, _ in v}) >= 2}
ordenadas = sorted(ambiguas.items(), key=lambda kv: -len({p for p, _, _ in kv[1]}))

filas = []
for palabra, apariciones in ordenadas[:6]:
    vistos = set()
    for pos, oracion, libro in apariciones:
        if pos in vistos:
            continue
        vistos.add(pos)
        ctx = " ".join(oracion.split())
        filas.append({"palabra": palabra, "POS": pos, "libro": libro,
                      "contexto": ctx[:95] + ("..." if len(ctx) > 95 else "")})

print(f"palabras con etiqueta ambigua en las muestras: {len(ambiguas):,}\n")
pd.DataFrame(filas).set_index(["palabra", "POS"])

¿Hay algún libro donde el tagger cometa visiblemente más errores? ¿A qué se debe?

Las categorías ya separan a los tres autores. Frankenstein tiene más sustantivos (16.48%) y menos puntuación (11.20%): prosa reflexiva con poco diálogo. Pride and Prejudice es lo contrario, 15.19% de puntuación y 4.08% de nombres propios. Dracula tiene más pronombres (14.79%), propio de la primera persona.

El tagger falla más en Austen, y la causa principal no es el inglés de 1813 sino la marcación de Gutenberg: en _you_ etiqueta el segundo guión bajo como pronombre. El otro error sí es de época: en They all paint tables, cover screens, and net purses marca paint como sustantivo y net como adjetivo, cuando las tres son verbos. Net con el sentido de tejer ya no se usa.

Dracula es el que menos sufre porque su sintaxis es la más cercana al inglés actual. Las palabras ambiguas confirman que la ambigüedad es estructural: that sale como SCONJ, PRON y DET; well como ADV, INTJ y ADJ.

## 3. Tagsets y anotación

Libro elegido: Pride and Prejudice, por ser el que más errores mostró.

In [ ]:
libro_anot = "Pride and Prejudice"
docs_pp = libros[libro_anot]["docs_muestra"]

muestra_tags = [
    {"token": t.text, "POS (universal)": t.pos_, "tag_ (Penn Treebank)": t.tag_,
     "significado del tag fino": spacy.explain(t.tag_)}
    for d in docs_pp for t in d
    if t.is_alpha and t.pos_ in {"NOUN", "VERB", "ADJ", "PRON", "ADV"}
][:10]

pd.DataFrame(muestra_tags)

¿Qué información adicional aporta Penn Treebank frente al tagset universal?

El universal dice qué clase de palabra es; Penn Treebank dice además en qué forma está.

En los diez ejemplos un solo VERB se reparte en VB, VBP, VBD y VBN. Esa distinción es la que permite saber si acknowledged en the acknowledged lovers es un pasado o un participio adjetivo. Lo mismo con NOUN, que se abre en NN y NNS.

Lo que se pierde es portabilidad. Penn Treebank codifica morfología inglesa y no sirve para comparar entre idiomas.

In [ ]:
anotacion_manual = {
 0: ["``","PRP","VBP","DT","NN",",","CC","VBP","TO","VB","DT",".","''"],
 1: ["PRP","MD","RB","RB","VB","IN","DT","NN","IN","PRP$","NNS","."],
 2: ["PRP","MD","VB","DT","NNS","IN","NFP","PRP","NFP","."],
 3: ["DT","JJ","NNS","VBD","CC","VBD",":","DT","JJ","VBD","JJ","."],
 4: ["PRP","DT","VBP","NNS",",","VBP","NNS",",","CC","VBP","NNS","."],
}

cortas = [o for o in libros[libro_anot]["muestra"] if 5 <= len(o.split()) <= 11][:5]

filas, total, aciertos = [], 0, 0
for k, oracion in enumerate(cortas):
    doc = nlp(oracion)
    auto = [t for t in doc if not t.is_space]
    manual = anotacion_manual[k]
    for tok, mi in zip(auto, manual):
        coincide = tok.tag_ == mi
        total += 1; aciertos += coincide
        if not coincide:
            filas.append({"oración": k, "token": tok.text,
                          "manual": mi, "tagger": tok.tag_, "": "≠"})

print(f"tokens comparados : {total}")
print(f"coincidencias     : {aciertos}  ({aciertos/total*100:.1f}%)")
print(f"desacuerdos       : {total - aciertos}\n")
pd.DataFrame(filas)

Comentario sobre los desacuerdos

Los cuatro son de tres tipos distintos.

El de _you_ es error del modelo, inducido por el corpus: etiqueta el guión bajo de cierre como pronombre y no hay lectura que lo justifique.

Los de paint y net son errores de dominio. Son verbos coordinados, pero el tagger elige el uso dominante en inglés moderno.

El de acknowledged es ambigüedad genuina. Manual lo marcó JJ por ser participio adjetivo, el tagger VBN conservando lo verbal. Penn Treebank admite las dos.

55 de 59 tokens coinciden, 93.2%. Tres de los cuatro desacuerdos son errores reales, así que la tasa efectiva ronda el 5%.

## 4. Reconocimiento de entidades nombradas (NER)

Se corre el modelo sobre los primeros 40,000 caracteres de cada novela, que equivalen a los primeros dos o tres capítulos.

In [ ]:
FRAGMENTO = 40_000
TIPOS_LUGAR = {"GPE", "LOC", "FAC"}

for titulo, info in libros.items():
    doc = nlp(info["texto"][:FRAGMENTO])
    info["doc_ner"] = doc
    info["ents"] = list(doc.ents)
    info["tokens_ner"] = len([t for t in doc if not t.is_space and not t.is_punct])

tipos = sorted({e.label_ for i in libros.values() for e in i["ents"]})
tabla_ner = pd.DataFrame({
    t: {tipo: sum(1 for e in i["ents"] if e.label_ == tipo) for tipo in tipos}
    for t, i in libros.items()
})
tabla_ner.loc["TOTAL"] = tabla_ner.sum()
tabla_ner.loc["tokens del fragmento"] = [libros[t]["tokens_ner"] for t in tabla_ner.columns]
tabla_ner.loc["densidad (ent./100 tok.)"] = (
    tabla_ner.loc["TOTAL"] / tabla_ner.loc["tokens del fragmento"] * 100).round(2)
tabla_ner

In [ ]:
for titulo, info in libros.items():
    personas = Counter(e.text.strip() for e in info["ents"] if e.label_ == "PERSON")
    lugares  = Counter(e.text.strip() for e in info["ents"] if e.label_ in TIPOS_LUGAR)
    print("=" * 74)
    print(f"{titulo}  —  {info['autor']}, {info['anio']}")
    print("=" * 74)
    print(f"  PERSON  top-10: {personas.most_common(10)}")
    print(f"  GPE/LOC top-10: {lugares.most_common(10)}")
    print()

¿Coinciden con los personajes y lugares esperados? ¿Qué dice la densidad?

En Austen el modelo recupera bien el elenco: Bingley, Bennet, Darcy, Jane y Elizabeth, con Netherfield y Meryton como lugares. Es donde mejor funciona, porque los nombres aparecen en diálogo y con títulos que los señalan.

Frankenstein tiene 2.17 entidades por cada 100 tokens contra 5.00 de Austen. Coincide con su 1.29% de nombres propios: el narrador nombra a los demás por parentesco y lo que aparece son topónimos europeos.

En Dracula acierta con los topónimos pero se equivoca más. Las palabras extranjeras y los sustantivos capitalizados por convención epistolar lo confunden, y varios topónimos salen como ORG en lugar de GPE.

La densidad separa los géneros: la novela de costumbres tiene más del doble de entidades por token que la filosófica.

## 5. Esquema BIO

Cinco oraciones consecutivas de Pride and Prejudice, con la secuencia BIO token por token a partir de las entidades del modelo.

In [ ]:
ors_pp = sent_tokenize(libros["Pride and Prejudice"]["texto"], language="english")
inicio = next(k for k, o in enumerate(ors_pp) if "Bennet" in o)
doc_bio = nlp(" ".join(ors_pp[inicio:inicio + 5]))

filas_bio = [{"token": t.text,
              "BIO": f"{t.ent_iob_}-{t.ent_type_}" if t.ent_iob_ in "BI" else "O"}
             for t in doc_bio if not t.is_space]

print(f"{len(filas_bio)} tokens\n")
print(doc_bio.text[:300], "...\n")

tabla_bio = pd.DataFrame(filas_bio).T
tabla_bio.columns = [""] * tabla_bio.shape[1]
tabla_bio

## 6. Evaluación de entidades: anotación manual y métricas

Las novelas no traen entidades anotadas, así que se construye la referencia a mano. Se leyeron las 30 primeras oraciones de cada libro y se marcaron las entidades verdaderas con su tipo y span exacto.

Criterios: los títulos de cortesía no son parte del nombre; ríos, cordilleras y regiones van como GPE; los gentilicios como NORP y no como PERSON; el pie de imprenta del editor no es entidad; fechas, horas y números quedan fuera.

In [ ]:
DESCARTAR = {"DATE", "TIME", "CARDINAL", "ORDINAL", "MONEY", "PERCENT", "QUANTITY"}

GOLD = {
    "Pride and Prejudice": {
        (84, 85, "PERSON"), (101, 103, "GPE"), (110, 111, "PERSON"),
        (129, 130, "PERSON"), (146, 147, "PERSON"), (231, 232, "PERSON"),
        (234, 235, "GPE"), (249, 250, "GPE"), (281, 282, "PERSON"),
        (384, 385, "PERSON"), (513, 514, "PERSON"),
    },
    "Frankenstein": {
        (62, 63, "GPE"), (73, 74, "GPE"), (182, 183, "PERSON"),
        (609, 613, "GPE"), (645, 647, "PERSON"), (780, 781, "PERSON"),
        (880, 883, "GPE"), (960, 961, "GPE"),
    },
    "Dracula": {
        (17, 18, "GPE"), (36, 39, "GPE"), (120, 121, "GPE"), (154, 155, "GPE"),
        (163, 166, "GPE"), (204, 205, "PERSON"), (246, 247, "GPE"),
        (285, 286, "GPE"), (292, 294, "ORG"), (308, 309, "GPE"),
        (364, 365, "GPE"), (366, 367, "GPE"), (368, 369, "GPE"),
        (375, 376, "GPE"), (388, 389, "GPE"), (409, 411, "GPE"),
        (428, 430, "ORG"), (436, 437, "GPE"), (445, 446, "PERSON"),
        (479, 480, "PERSON"), (485, 486, "GPE"), (503, 504, "NORP"),
        (520, 521, "NORP"), (542, 543, "PERSON"), (545, 546, "NORP"),
        (567, 568, "NORP"), (590, 591, "GPE"), (868, 869, "GPE"),
    },
}


def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f


resultados = []
for titulo, info in libros.items():
    ors = sent_tokenize(info["texto"], language="english")
    doc = nlp(" ".join(ors[:30]))

    pred = {(e.start, e.end, e.label_) for e in doc.ents if e.label_ not in DESCARTAR}
    gold = GOLD[titulo]

    tp = len(pred & gold); fp = len(pred - gold); fn = len(gold - pred)
    p, r, f = prf(tp, fp, fn)
    resultados.append({"libro": titulo, "evaluación": "entidad completa",
                       "precisión": round(p, 3), "recall": round(r, 3), "F1": round(f, 3),
                       "TP": tp, "FP": fp, "FN": fn})

    tok_pred = {(i, lab) for ini, fin, lab in pred for i in range(ini, fin)}
    tok_gold = {(i, lab) for ini, fin, lab in gold for i in range(ini, fin)}
    tp = len(tok_pred & tok_gold); fp = len(tok_pred - tok_gold); fn = len(tok_gold - tok_pred)
    p, r, f = prf(tp, fp, fn)
    resultados.append({"libro": titulo, "evaluación": "token",
                       "precisión": round(p, 3), "recall": round(r, 3), "F1": round(f, 3),
                       "TP": tp, "FP": fp, "FN": fn})

tabla_eval = pd.DataFrame(resultados).set_index(["libro", "evaluación"]).sort_index()
tabla_eval

¿La evaluación por token da una impresión más optimista? ¿Funciona mejor en algún libro?

No, y contradice lo esperable. En dos de los tres libros la de token es más severa: Austen baja de 0.750 a 0.667 y Shelley de 0.625 a 0.429. Solo Dracula se invierte, de 0.328 a 0.341.

Depende del largo de las entidades. El argumento de que token es más benévola vale para errores de frontera: si predice Count Dracula y la referencia dice Dracula, por entidad son dos fallos y por token uno se acredita. Se invierte con las entidades inventadas largas: tres tokens marcados como ORG cuentan como un falso positivo por entidad y tres por token. Acá predominan los segundos.

Ninguna métrica es uniformemente más alta, y comparar las dos es lo que dice qué error domina.

El modelo anda mejor en Austen, 0.750. Dracula es el peor con 0.328, y no porque no detecte entidades sino porque las tipifica mal: Buda-Pesth y Danube como personas, Bistritz y Mina como organizaciones, Attila como lugar. Shelley queda en 0.625, con pocas entidades y por lo tanto pocas ocasiones de fallar.

## 7. Conclusiones comparativas: épocas y autores

Diferencias de estilo

Las categorías describen a cada autor sin leer una línea. Austen es la más dialogada, 15.19% de puntuación y 4.08% de nombres propios. Shelley es lo opuesto, 16.48% de sustantivos y 1.29% de nombres propios, con un narrador que reflexiona en abstracto. Stoker queda en medio, más pronombres y menos adjetivos. La densidad de entidades dice lo mismo: donde hay más diálogo hay más nombres propios.

Ambigüedad POS y modelos n-grama

Son el mismo problema con distinta salida. Un n-grama estima la probabilidad de una palabra dadas las anteriores; un tagger, la de una etiqueta dada la palabra y su contexto. En los dos la respuesta está en lo que rodea al elemento, no en él, y en los dos la evidencia sale de contar sobre un corpus. Cuando el tagger duda entre SCONJ, PRON y DET para that hace lo mismo que el bigrama del laboratorio anterior al elegir la siguiente palabra.

Desfase de dominio y OOV

Tres tipos de error. Verbos con acepciones en desuso, como net. Topónimos extranjeros que tipifica mal, que es lo que hunde a Dracula. Y capitalización epistolar que confunde con nombres propios.

Es el mismo OOV del laboratorio anterior, corrido de lugar. Allá una palabra fuera del vocabulario anulaba la oración entera. Acá una palabra fuera del dominio recibe la etiqueta más frecuente en inglés moderno, que suele estar mal. El modelo de lenguaje avisa declarando el imposible; el tagger se equivoca callado.

Qué métrica reportar

La de entidad completa, pero no por el motivo habitual. El argumento típico es que la de token infla el número, y acá pasó lo contrario en dos de tres. La razón real es que mide lo que el sistema entrega: si alguien pide nombres de personas, Count Dracula cuando la respuesta era Dracula está mal, no medio bien. La unidad de la métrica tiene que ser la del producto. La de token sirve puertas adentro, para saber si los fallos son de frontera o si el modelo inventa entidades largas.

Las tres novelas apuntan a lo mismo. Estas herramientas funcionan sobre textos de hace dos siglos pero se degradan según cuánto se aleje el texto del inglés con el que aprendieron, y esa distancia no es cronológica. Dracula es la más reciente y la que peor anda, 0.328, por sus nombres de Europa del Este. Pride and Prejudice es la más vieja y la mejor, 0.750, porque su vocabulario sigue siendo reconocible. Pesa la distancia al corpus de entrenamiento, no la antigüedad.